# 11 Reflexión Final:

### a) ¿Por qué la reconstrucción tomográfica se considera un problema inverso?

* **Problema directo vs problema inverso:**
  * Un **problema directo** es un problema donde se conoce el estado interno del objeto o paciente (la imagen $x$) y se conoce cómo opera el escáner (la matriz $A$). Con esos dos datos, se puede calcular qué mediciones registrarán los sensores haciendo simplemente una multiplicación ($b = Ax$).
  * Por el contrario, la reconstrucción tomográfica es un **problema inverso** porque en este caso ocurre lo opuesto, no podemos ver el interior del paciente. Lo que tenemos son las mediciones registradas por los detectores ($b$) y la "forma" de los rayos ($A$), y nuestro objetivo es "ir para atrás", o sea deducir la distribución interna de tejidos ($x$) a partir de los datos observados. 

* **Variables del problema:**
  * **Información conocida:** Se conoce al vector de mediciones del escáner $b \in \mathbb{R}^m$ y la matriz de proyección $A \in \mathbb{R}^{m \times n^2}$ (que describe qué pixeles atraviesa cada rayo).
  * **Variable que se busca recuperar:** El vector $x \in \mathbb{R}^{n^2}$, que representa los coeficientes de atenuación de cada píxel de la imagen que queremos reconstruir.

* **El papel del modelo $b = Ax$:**
  Conecta la física con la computación. Nace de aplicar log a: $-\ln(I/I_0) = \int \mu \, dr$. Esto nos permite modelar cada medición del sensor como una suma de atenuaciones a lo largo del rayo, haciendo todo el proceso como un sistema de ecuaciones lineales estándar.

### b) ¿Cómo puede determinarse, a partir de $A$, si las mediciones permiten recuperar una única imagen?

* **Condición de unicidad:**
  Para que una imagen pueda recuperarse de manera única, no pueden existir ambigüedades en el sistema. En términos de la materia, esto exige que el **espacio nulo de $A$ sea trivial**:
  $$\dim(\operatorname{Nul}(A)) = 0$$
  Por el Teorema de la Dimensión:
  $$\operatorname{rango}(A) + \dim(\operatorname{Nul}(A)) = n^2 = 64$$
  Por lo tanto, la reconstrucción es única si y solo si la matriz tiene rango completo de columnas: $\operatorname{rango}(A) = n^2 = 64$.

* **Lo que observamos en el trabajo:**
  * Para $A_2$ (16 rayos): $\operatorname{rango}(A_2) = 15 \implies \dim(\operatorname{Nul}(A_2)) = 49$.
  * Para $A_4$ (44 rayos): $\operatorname{rango}(A_4) = 27 \implies \dim(\operatorname{Nul}(A_4)) = 37$.
  * Para $A_7$ (119 rayos): $\operatorname{rango}(A_7) = 43 \implies \dim(\operatorname{Nul}(A_7)) = 21$.
  
  En ningun caso la solución es única! Incluso con 119 mediciones para 64 incógnitas, muchos rayos son linealmente dependientes entre sí y el rango nunca llega a 64.

* **¿Qué ocurre si existe un vector $v \neq 0$ tal que $Av = 0$?**
  Si tenemos una imagen candidata $x$ que explica las mediciones ($Ax = b$), podemos construir otra imagen sumándole cualquier múltiplo de $v$:
  $$x_{\text{alt}} = x + \gamma v$$
  Al pasar por el escáner:
  $$A x_{\text{alt}} = A(x + \gamma v) = Ax + \gamma Av = b + 0 = b$$
  El tomógrafo genera **exactamente las mismas mediciones** para ambas imágenes. Físicamente, el vector $v$ representa un "patrón invisible".

### c) ¿Cómo afecta el ruido al sistema de ecuaciones?

* **Impacto del ruido:**
  En condiciones reales, los detectores sufren ruido (por ejemplo interferencia), por lo que medimos $b_\sigma = b + \text{ruido}$. Este ruido "empuja" al vector de mediciones fuera del espacio generado por las columnas de $A$:
  $$b_\sigma \notin \operatorname{Col}(A)$$
  Como consecuencia, el sistema $Ax = b_\sigma$ se vuelve incompatible. No existe ninguna imagen que cumpla exactamente todas las ecuaciones a la vez.

* **¿Qué problema resuelve mínimos cuadrados?**
  Como no hay solución exacta, mínimos cuadrados busca la "mejor aproximación posible. Encuentra el vector $\hat{x}$ cuya predicción $A\hat{x}$ esté a la menor distancia posible de las mediciones ruidosas $b_\sigma$, minimizando la función de costo:
  $$E(x) = \frac{1}{2} \|Ax - b_\sigma\|_2^2$$

* **Interpretación del residuo:**
  El residuo $r = A\hat{x} - b_\sigma$ representa la diferencia entre lo que la imagen reconstruida proyecta en los detectores y lo que el escáner realmente registró. En el ej. 7 demostramos que este residuo es perpendicular al espacio columna de $A$ ($A^T r = 0$), lo que confirma que $A\hat{x}$ es la proyección ortogonal de $b_\sigma$ sobre $\operatorname{Col}(A)$.

* **¿La norma del residuo es suficiente para evaluar la calidad?**
  No, no lo es. La norma del residuo $\|A\hat{x} - b_\sigma\|$ solo mide qué tan bien coinciden las ecuaciones en el espacio de detectores, pero no nos dice nada de la calidad de la imagen.
  En el ej. 6 vimos que a medida que aumenta el ruido, mínimos cuadrados logra hacer su residuo pequeño. Pero la imagen resultante queda llena de artefactos granulares, con un error muy alto respecto al fantoma real e incluso con valores de píxeles negativos, que son imposibles.

### d) ¿Qué supuesto sobre la imagen introduce el término $\|Dx\|^2$?

* **Supuesto introducido:**
  La matriz $D = \begin{pmatrix} D_h \\ D_v \end{pmatrix}$ calcula las diferencias entre pixeles vecinos (horizontales y verticales). Por lo tanto, el término $\|Dx\|^2 = \sum (x_i - x_j)^2$ mide la falta de suavidad de la imagen.
  Al agregar este término al costo con un peso $\lambda$ introducimos el supuesto de que los tejidos del cuerpo humano son en su mayoría continuos, suaves y homogéneos y que los saltos bruscos aleatorios pixel a pixel se deben al ruido y no al paciente.

* **Comportamiento al modificar $\lambda$:**
  * **Con $\lambda = 0$:** La imagen reconstruida queda muy ruidosa, con pixeles negativos.
  * **Con $\lambda$ moderado (de 0.1 a 0.3):** Es el balance óptimo. La penalización suprime las variaciones bruscas del ruido sin borrar las estructuras reales del objeto, llegando a contornos limpios y nitidos y el menor error respecto al fantoma.
  * **Con $\lambda$ muy grande ($\ge 1.0$):** El costo le da un peso excesivo a la suavidad. La penalización aplasta todas las diferencias, borrando los bordes de los órganos internos y volviendo a la imagen en una mancha homogenea y borrosa.

### e) Comparación entre `np.linalg.lstsq`, gradiente descendiente y Newton

| Criterio | `np.linalg.lstsq` | Gradiente Descendiente | Método de Newton |
| :--- | :--- | :--- | :--- |
| **Tipo de método** | Directo (álgebra lineal) | Iterativo | Iterativo |
| **Iteraciones** | 1 cálculo directo (basado en SVD) | Muchas | Una iteración |
| **Parámetros requeridos** | solo $A$ y $b$ | Tasa de aprendizaje $\alpha$ y criterio de parada | Ninguno para el paso (resuelve $Hp = g$) |
| **Uso de memoria** | Moderado a alto | Pequeño (no almacena matrices densas) | Muy alto (requiere la matriz Hessiana $H$) |
| **Solución obtenida** | Solución de norma mínima | Solución idéntica a `lstsq` al converger | Solución idéntica a `lstsq` |

* **Comportamiento Observado para Newton**
    El mínimo de la función de costo ocurre justo donde se anula el gradiente: $\nabla E(x) = 0$. Si desarrollamos la fórmula del gradiente que nos da el enunciado:
    $$\nabla E(x) = A^T(Ax - b) = A^T A x - A^T b = H x - A^T b$$
    Vemos que es una función lineal de $x$, y que la Hessiana $H = A^T A$ es constante (no cambia según el punto donde nos paremos)
    Si estamos en un punto actual $x_t$ y nos movemos a $x_{t+1} = x_t - p_t$, el nuevo gradiente se calcula asi:
    $$\nabla E(x_t - p_t) = H(x_t - p_t) - A^T b = (H x_t - A^T b) - H p_t = \nabla E(x_t) - H p_t$$
    Para que en este punto el gradiente sea cero y llegar al mínimo en ese paso:
    $$\nabla E(x_t) - H p_t = 0 \iff H p_t = \nabla E(x_t)$$
    Como el método de Newton define el paso $p_t$ resolviendo ese sistema lineal ($H p_t = \nabla E(x_t)$), el paso da justo y anula el gradiente en el primer intento. Por eso, como es una función cuadrática con Hessiana constante, converge en una sola iteración.

### f) Síntesis Integradora


1. **El modelo $b = Ax$:**
   Permite plantear la absorción de rayos X como un sistema de ecuaciones lineales. Cada fila representa la trayectoria de un rayo y cada columna representa un píxel de la imagen.

2. **El Rango y el Espacio Nulo:**
   Permiten analizar la unicidad de la solución. El rango indica cuánta información independiente capturan los rayos, mientras que el espacio nulo describe los patrones "ciegos" que el tomógrafo no puede detectar ($Av = 0$).

3. **Mínimos Cuadrados y Proyecciones Ortogonales:**
   Permiten resolver la incompatibilidad frente al ruido. Cuando los sensores tienen ruido, las mediciones no pertenecen al espacio columna ($b_\sigma \notin \operatorname{Col}(A)$). Mínimos cuadrados encuentra la mejor aproximación posible mediante la proyección ortogonal sobre $\operatorname{Col}(A)$, esto asegura que el vector residual sea perpendicular a las columnas de la matriz ($A^T r = 0$).

4. **La Regularización:**
   Nos sirve para limpiar el ruido sin arruinar la imagen:
   - Mínimos cuadrados solo intenta ajustar los datos, por lo que copia todo el ruido.
   - Al agregarle el término ${\lambda}\|Dx\|^2$ , le pedimos a la imagen que sea suave entre píxeles vecinos. Con ${\lambda}$ buscamos el punto justo entre sacar el ruido y no borrar los detalles importantes del objeto.